### 1. Check Execution Environment

Check the GPU and CUDA environment for knowledge graph embedding training.

In [2]:
import torch

print(f'[INFO] CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'[INFO] Current GPU device: {torch.cuda.get_device_name(0)}')
    print(f'[INFO] CUDA version: {torch.version.cuda}')
    device = torch.device('cuda')
    print(f'[DONE] Using device: {device}')
else:
    device = torch.device('cpu')
    print(f'[WARNING] CUDA is not available. Using device: {device}')

[INFO] CUDA available: True
[INFO] Current GPU device: NVIDIA GeForce RTX 3080
[INFO] CUDA version: 12.4
[DONE] Using device: cuda


### 2. Load the Final Knowledge Graph

Load the saved final knowledge graph file and convert it into a triple list. These triples will be used for data splitting in closed-world evaluation.

In [3]:
import pandas as pd
from pathlib import Path

current_dir = Path.cwd().resolve()
BASE_DIR = next(path for path in [current_dir, *current_dir.parents] if (path / 'data').exists())

kg_file = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12' / 'nvd_threat_kg.csv'

print(f'[INFO] Processing {kg_file.name} ...')

triples_df = pd.read_csv(kg_file)
triples = triples_df[['subject', 'predicate', 'object']].values.tolist()

print(f'[DONE] Knowledge graph loaded successfully ({len(triples):,} triples)')

[INFO] Processing nvd_threat_kg.csv ...
[DONE] Knowledge graph loaded successfully (1,706,087 triples)


### 3. Split Data for Closed-World Evaluation

Randomly split the loaded knowledge graph triples into training, validation, and test sets for closed-world evaluation. The validation and test sets each contain 10,000 triples.

In [7]:
import numpy as np
from pykeen.triples import TriplesFactory

triples_array = np.asarray(triples, dtype=str)

tf = TriplesFactory.from_labeled_triples(
    triples_array,
    create_inverse_triples=False
)

valid_size = 10000
test_size = 10000

valid_ratio = valid_size / tf.num_triples
test_ratio = test_size / tf.num_triples
train_ratio = 1.0 - valid_ratio - test_ratio

train_factory, valid_factory, test_factory = tf.split(
    [train_ratio, valid_ratio, test_ratio],
    random_state=0
)

X_train = train_factory.triples
X_valid = valid_factory.triples
X_test = test_factory.triples

print(f'[DONE] Triple split completed successfully')
print(f'[INFO] Total triples: {tf.num_triples:,}')
print(f'[INFO] Train triples: {train_factory.num_triples:,}')
print(f'[INFO] Validation triples: {valid_factory.num_triples:,}')
print(f'[INFO] Test triples: {test_factory.num_triples:,}')

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


[DONE] Triple split completed successfully
[INFO] Total triples: 1,705,929
[INFO] Train triples: 1,685,929
[INFO] Validation triples: 10,000
[INFO] Test triples: 10,000


### 4. Save Split Datasets

Save the training, validation, and test sets created for closed-world evaluation as CSV files. The filenames indicate that the snapshot includes data up to December 2024.

In [8]:
output_dir = BASE_DIR / 'data' / 'processed' / 'embedding' / '2024_12'
output_dir.mkdir(parents=True, exist_ok=True)

train_file = output_dir / 'nvd_threat_kg_train.csv'
valid_file = output_dir / 'nvd_threat_kg_valid.csv'
test_file = output_dir / 'nvd_threat_kg_test.csv'

train_df = pd.DataFrame(X_train, columns=['subject', 'predicate', 'object'])
valid_df = pd.DataFrame(X_valid, columns=['subject', 'predicate', 'object'])
test_df = pd.DataFrame(X_test, columns=['subject', 'predicate', 'object'])

train_df.to_csv(train_file, index=False, encoding='utf-8-sig')
valid_df.to_csv(valid_file, index=False, encoding='utf-8-sig')
test_df.to_csv(test_file, index=False, encoding='utf-8-sig')

print(f'[INFO] Output directory: {output_dir}')
print(f'[DONE] Saved: {train_file} ({len(train_df):,} rows)')
print(f'[DONE] Saved: {valid_file} ({len(valid_df):,} rows)')
print(f'[DONE] Saved: {test_file} ({len(test_df):,} rows)')

[INFO] Output directory: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12
[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\nvd_threat_kg_train.csv (1,685,929 rows)
[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\nvd_threat_kg_valid.csv (10,000 rows)
[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\nvd_threat_kg_test.csv (10,000 rows)


### 5. Reload Split Datasets

Reload the saved training, validation, and test datasets and convert them into `TriplesFactory` objects for PyKEEN training and evaluation. This step verifies that the saved datasets were created correctly and prepares the input format for embedding training.

In [4]:
import pandas as pd
import numpy as np
from pykeen.triples import TriplesFactory

input_dir = BASE_DIR / 'data' / 'processed' / 'embedding' / '2024_12'

train_file = input_dir / 'nvd_threat_kg_train.csv'
valid_file = input_dir / 'nvd_threat_kg_valid.csv'
test_file = input_dir / 'nvd_threat_kg_test.csv'

print(f'[INFO] Processing {train_file.name} ...')
train_df = pd.read_csv(train_file)

print(f'[INFO] Processing {valid_file.name} ...')
valid_df = pd.read_csv(valid_file)

print(f'[INFO] Processing {test_file.name} ...')
test_df = pd.read_csv(test_file)

X_train = train_df[['subject', 'predicate', 'object']].to_numpy(dtype=str)
X_valid = valid_df[['subject', 'predicate', 'object']].to_numpy(dtype=str)
X_test = test_df[['subject', 'predicate', 'object']].to_numpy(dtype=str)

training_factory = TriplesFactory.from_labeled_triples(
    X_train,
    create_inverse_triples=False
)

validation_factory = TriplesFactory.from_labeled_triples(
    X_valid,
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id,
    create_inverse_triples=False
)

testing_factory = TriplesFactory.from_labeled_triples(
    X_test,
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id,
    create_inverse_triples=False
)

print(f'[DONE] Split datasets loaded successfully')
print(f'[INFO] Train triples: {X_train.shape[0]:,}')
print(f'[INFO] Validation triples: {X_valid.shape[0]:,}')
print(f'[INFO] Test triples: {X_test.shape[0]:,}')

c:\Users\SSRC\miniconda3\envs\threat-kg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Processing nvd_threat_kg_train.csv ...
[INFO] Processing nvd_threat_kg_valid.csv ...
[INFO] Processing nvd_threat_kg_test.csv ...
[DONE] Split datasets loaded successfully
[INFO] Train triples: 1,685,929
[INFO] Validation triples: 10,000
[INFO] Test triples: 10,000


### 6. Train and Evaluate the TransE Model

Train the TransE model using PyKEEN and evaluate its performance on the validation and test datasets. After training, check the main evaluation metrics and save the trained model.

In [ ]:
import torch
from pykeen.pipeline import pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Using device: {device}')

result_transe = pipeline(
    training=training_factory,
    validation=validation_factory,
    testing=testing_factory,
    model='TransE',
    model_kwargs=dict(
        embedding_dim=200,
        scoring_fct_norm=1,
    ),
    training_loop='sLCWA',
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=20,
        corruption_scheme=('head', 'tail'),
    ),
    training_kwargs=dict(
        num_epochs=300,
        batch_size=4096,
        use_tqdm=True,
    ),
    optimizer='adam',
    optimizer_kwargs=dict(lr=1e-4),
    loss='nssa',
    regularizer='lp',
    regularizer_kwargs=dict(
        p=3,
        weight=1e-5,
    ),
    device=device,
    random_seed=2,
)

metric_results = result_transe.metric_results

try:
    mrr = metric_results.get_metric('mrr')
    hits10 = metric_results.get_metric('hits@10')
except Exception:
    metric_dict = metric_results.to_flat_dict()
    mrr = metric_dict.get('both.realistic.inverse_harmonic_mean_rank')
    hits10 = metric_dict.get('both.realistic.hits_at_10')

print('=' * 40)
print('[DONE] TransE training completed successfully')
print(f'[INFO] MRR: {mrr:.4f}')
print(f'[INFO] Hits@10: {hits10:.4f}')
print('=' * 40)

model_dir = BASE_DIR / 'models' / '2024_12' / 'transe-seed-2'
model_dir.mkdir(parents=True, exist_ok=True)

result_transe.save_to_directory(model_dir)
print(f'[DONE] Saved: {model_dir}')

[INFO] Using device: cuda


Training epochs on cuda:0: 100%|██████████| 300/300 [1:04:55<00:00, 12.98s/epoch, loss=0.1, prev_loss=0.1]    
Evaluating on cuda:0: 100%|██████████| 10.0k/10.0k [06:51<00:00, 24.3triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 498.16s seconds


[DONE] TransE training completed successfully
[INFO] MRR: 0.4553
[INFO] Hits@10: 0.6340


INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=446240, num_relations=18, create_inverse_triples=False, num_triples=1685929) to file:///C:/Workspace/icsa-threat-kg/models/2024_12/transe-seed-2/training_triples
INFO:pykeen.pipeline.api:Saved to directory: C:\Workspace\icsa-threat-kg\models\2024_12\transe-seed-2


[DONE] Saved: C:\Workspace\icsa-threat-kg\models\2024_12\transe-seed-2


### 7. Load the Trained TransE Model

Load the previously trained TransE model from the saved directory and move it to the available device. After loading, check the model structure and the total number of parameters to confirm that the model was restored correctly.

In [5]:
import torch

model_path = BASE_DIR / 'models' / '2024_12' / 'transe-seed-1' / 'trained_model.pkl'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Using device: {device}')

if not model_path.exists():
    raise FileNotFoundError(f'[ERROR] Model file not found: {model_path}')

model_transe = torch.load(model_path, map_location=device, weights_only=False)
model_transe = model_transe.to(device)
model_transe.eval()

print('=' * 40)
print('[DONE] TransE model loaded successfully')
print(f'[INFO] Model: {model_transe.__class__.__name__}')
print(f'[INFO] Number of parameters: {sum(p.numel() for p in model_transe.parameters()):,}')
print(f'[INFO] Loaded from: {model_path}')
print('=' * 40)

[INFO] Using device: cuda
[DONE] TransE model loaded successfully
[INFO] Model: TransE
[INFO] Number of parameters: 89,251,600
[INFO] Loaded from: C:\Workspace\icsa-threat-kg\models\2024_12\transe-seed-1\trained_model.pkl


### 8. Re-evaluate the Trained TransE Model with Additional Ranking Metrics

Re-evaluate the trained TransE model on the test set to obtain additional rank-based metrics such as MR, MRR, Hits@1, Hits@3, Hits@10, and Hits@20. During filtered evaluation, use the training and validation triples as additional filter triples so that all known true triples are considered properly.

In [7]:
from pykeen.evaluation import RankBasedEvaluator

print('[INFO] Re-evaluating TransE model on the test set ...')

evaluator = RankBasedEvaluator(
    metrics=['hits@k'],
    metrics_kwargs=[{'k': 20}],
    add_defaults=True
)

evaluation_results = evaluator.evaluate(
    model=model_transe,
    mapped_triples=testing_factory.mapped_triples,
    additional_filter_triples=[
        training_factory.mapped_triples,
        validation_factory.mapped_triples
    ],
    batch_size=2048,
    device=device
)

try:
    mrr = evaluation_results.get_metric('mrr')
    mr = evaluation_results.get_metric('mean_rank')
    hits20 = evaluation_results.get_metric('hits@20')
    hits10 = evaluation_results.get_metric('hits@10')
    hits3 = evaluation_results.get_metric('hits@3')
    hits1 = evaluation_results.get_metric('hits@1')
except Exception:
    metric_dict = evaluation_results.to_flat_dict()
    mrr = metric_dict.get('both.realistic.inverse_harmonic_mean_rank')
    mr = metric_dict.get('both.realistic.arithmetic_mean_rank')
    hits20 = metric_dict.get('both.realistic.hits_at_20')
    hits10 = metric_dict.get('both.realistic.hits_at_10')
    hits3 = metric_dict.get('both.realistic.hits_at_3')
    hits1 = metric_dict.get('both.realistic.hits_at_1')

print('=' * 40)
print('[DONE] TransE evaluation completed successfully')
print(f'[INFO] MRR: {mrr:.3f}')
print(f'[INFO] MR: {mr:.3f}')
print(f'[INFO] Hits@20: {hits20:.3f}')
print(f'[INFO] Hits@10: {hits10:.3f}')
print(f'[INFO] Hits@3: {hits3:.3f}')
print(f'[INFO] Hits@1: {hits1:.3f}')
print('=' * 40)

[INFO] Re-evaluating TransE model on the test set ...


Evaluating on cuda:0: 100%|██████████| 10.0k/10.0k [07:22<00:00, 22.6triple/s]

[DONE] TransE evaluation completed successfully
[INFO] MRR: 0.458
[INFO] MR: 3546.965
[INFO] Hits@20: 0.699
[INFO] Hits@10: 0.637
[INFO] Hits@3: 0.511
[INFO] Hits@1: 0.363


### 9. Extract Relation-Specific Test Triples

Extract the relation-specific test triples for later evaluation. In this step, separate the test triples for MatchingCVE and MatchingCWE so that CVE→CPE and CVE→CWE evaluations can be performed in the following cells.

In [6]:
matching_cve_id = training_factory.relation_to_id['MatchingCVE']
matching_cwe_id = training_factory.relation_to_id['MatchingCWE']

mask_matching_cve = testing_factory.mapped_triples[:, 1] == matching_cve_id
mask_matching_cwe = testing_factory.mapped_triples[:, 1] == matching_cwe_id

test_cpe_cve_triples = testing_factory.mapped_triples[mask_matching_cve]
test_cve_cwe_triples = testing_factory.mapped_triples[mask_matching_cwe]

print('=' * 40)
print('[DONE] Target test triples extracted successfully')
print(f'[INFO] MatchingCVE relation ID: {matching_cve_id}')
print(f'[INFO] MatchingCWE relation ID: {matching_cwe_id}')
print(f'[INFO] Number of CPE-CVE test triples: {len(test_cpe_cve_triples):,}')
print(f'[INFO] Number of CVE-CWE test triples: {len(test_cve_cwe_triples):,}')
print('=' * 40)

[DONE] Target test triples extracted successfully
[INFO] MatchingCVE relation ID: 7
[INFO] MatchingCWE relation ID: 8
[INFO] Number of CPE-CVE test triples: 7,853
[INFO] Number of CVE-CWE test triples: 135


### 10. Define a Memory-Efficient Custom Evaluation Function for CVE→CPE Head Prediction

Define a memory-efficient custom evaluation function for head prediction on a restricted subset of entities. Instead of scoring all possible head entities and selecting a subset afterward, this function directly scores only the connected CPE candidates using PyKEEN’s restricted head scoring interface. This reduces GPU memory usage during CVE→CPE evaluation.

In [ ]:
import torch
import numpy as np
from collections import defaultdict

def evaluate_cve_to_cpe(model, test_triples, filter_triples, entities_subset,
                        device, batch_size, slice_size):

    # (1) Move the Model to the Target Device and Set Evaluation Mode
    model = model.to(device)
    model.eval()

    # (2) Convert filter_triples to a Unified NumPy Format
    if isinstance(filter_triples, torch.Tensor):
        filter_triples_np = filter_triples.detach().cpu().numpy()
    else:
        filter_triples_np = np.asarray(filter_triples)

    # (4) Prepare the Candidate Head Entity Subset
    if isinstance(entities_subset, torch.Tensor):
        candidate_heads = entities_subset.to(device)
    else:
        candidate_heads = torch.tensor(entities_subset, dtype=torch.long, device=device)

    if candidate_heads.numel() == 0:
        raise ValueError('[ERROR] entities_subset is empty.')

    # (3) Build a Mapping from (relation, tail) to True Head Entities
    rt_to_true_heads = defaultdict(set)
    for h, r, t in filter_triples_np:
        rt_to_true_heads[(int(r), int(t))].add(int(h))

    candidate_heads_list = candidate_heads.detach().cpu().tolist()
    candidate_head_to_pos = {entity_id: idx for idx, entity_id in enumerate(candidate_heads_list)}

    test_triples = test_triples.to(device)
    ranks = []

    with torch.inference_mode():
        total = len(test_triples)

        # (5) Process the Test Triples in Batches
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            batch = test_triples[start:end]

            # (6) Extract (relation, tail) Pairs from Each Batch
            true_heads = batch[:, 0]
            rels = batch[:, 1]
            tails = batch[:, 2]

            rt_batch = torch.stack([rels, tails], dim=1)

            # (7) Compute Scores for Candidate Head Entities
            subset_scores = model.predict_h(rt_batch, heads=candidate_heads, slice_size=slice_size)

            for i in range(batch.shape[0]):
                h = int(true_heads[i].item())
                r = int(rels[i].item())
                t = int(tails[i].item())

                if h not in candidate_head_to_pos:
                    raise ValueError(f'[ERROR] True head entity ID {h} is not included in entities_subset.')

                scores_i = subset_scores[i].clone()
                true_heads_for_rt = rt_to_true_heads[(r, t)]

                # (8) Filter Out Other True Head Entities
                filter_positions = [
                    candidate_head_to_pos[cand_h]
                    for cand_h in true_heads_for_rt
                    if cand_h != h and cand_h in candidate_head_to_pos
                ]

                if filter_positions:
                    filter_positions = torch.tensor(filter_positions, dtype=torch.long, device=device)
                    scores_i[filter_positions] = -torch.inf

                true_pos = candidate_head_to_pos[h]
                true_score = scores_i[true_pos]

                # (9) Identify the True Head Score and Compute Its Rank
                optimistic_rank = 1 + torch.sum(scores_i > true_score).item()
                pessimistic_rank = torch.sum(scores_i >= true_score).item()
                realistic_rank = 0.5 * (optimistic_rank + pessimistic_rank)

                ranks.append(realistic_rank)

    return np.asarray(ranks, dtype=np.float64)

### 11. Evaluate CVE→CPE Triples Using Connected CPE Candidates

Evaluate the trained TransE model on the CVE→CPE target triples in the cybersecurity knowledge graph. Since the MatchingCVE relation is stored as <CPE, MatchingCVE, CVE>, the evaluation is performed as a head prediction task. To make the evaluation more focused, use only the connected CPE nodes prepared during preprocessing as candidate entities, and apply filtered ranking with all known true triples.

In [12]:
connected_cpe_file = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12' / 'connected_cpe_list.txt'

if not connected_cpe_file.exists():
    raise FileNotFoundError(f'[ERROR] Connected CPE list not found: {connected_cpe_file}')

with open(connected_cpe_file, 'r', encoding='utf-8') as f:
    connected_cpe_names = [line.strip() for line in f if line.strip()]

connected_cpe_ids = sorted(
    training_factory.entity_to_id[cpe_name]
    for cpe_name in connected_cpe_names
    if cpe_name in training_factory.entity_to_id
)

missing_connected_cpe = [
    cpe_name
    for cpe_name in connected_cpe_names
    if cpe_name not in training_factory.entity_to_id
]

all_true_triples = torch.cat(
    [
        training_factory.mapped_triples,
        validation_factory.mapped_triples,
        testing_factory.mapped_triples
    ],
    dim=0
)

print('=' * 40)
print('[INFO] Preparing CVE->CPE evaluation ...')
print(f'[INFO] Connected CPE list file: {connected_cpe_file}')
print(f'[INFO] Number of connected CPE names: {len(connected_cpe_names):,}')
print(f'[INFO] Number of connected CPE IDs: {len(connected_cpe_ids):,}')
print(f'[INFO] Number of missing CPE names: {len(missing_connected_cpe):,}')
print(f'[INFO] Number of CPE-CVE test triples: {len(test_cpe_cve_triples):,}')
print(f'[INFO] Number of filter triples: {len(all_true_triples):,}')
print('=' * 40)

ranks_cve_to_cpe = evaluate_cve_to_cpe(model_transe, test_cpe_cve_triples, all_true_triples, connected_cpe_ids, 
                                        device='cuda', batch_size=128, slice_size=2048)

mrr = np.mean(1.0 / ranks_cve_to_cpe)
mr = np.mean(ranks_cve_to_cpe)
hits20 = np.mean(ranks_cve_to_cpe <= 20)
hits10 = np.mean(ranks_cve_to_cpe <= 10)
hits3 = np.mean(ranks_cve_to_cpe <= 3)
hits1 = np.mean(ranks_cve_to_cpe <= 1)

print('=' * 40)
print('[DONE] CVE->CPE evaluation completed successfully')
print(f'[INFO] MRR: {mrr:.3f}')
print(f'[INFO] MR: {int(mr):,}')
print(f'[INFO] Hits@20: {hits20:.3f}')
print(f'[INFO] Hits@10: {hits10:.3f}')
print(f'[INFO] Hits@3: {hits3:.3f}')
print(f'[INFO] Hits@1: {hits1:.3f}')
print('=' * 40)

[INFO] Preparing CVE->CPE evaluation ...
[INFO] Connected CPE list file: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\connected_cpe_list.txt
[INFO] Number of connected CPE names: 121,760
[INFO] Number of connected CPE IDs: 121,760
[INFO] Number of missing CPE names: 0
[INFO] Number of CPE-CVE test triples: 7,853
[INFO] Number of filter triples: 1,705,929
[DONE] CVE->CPE evaluation completed successfully
[INFO] MRR: 0.492
[INFO] MR: 560
[INFO] Hits@20: 0.742
[INFO] Hits@10: 0.665
[INFO] Hits@3: 0.525
[INFO] Hits@1: 0.409


### 12. Define a Memory-Efficient Custom Evaluation Function for CVE→CWE Tail Prediction

Define a custom evaluation function for tail prediction on a restricted subset of entities. This function is used to evaluate CVE→CWE triples by ranking only candidate CWE entities and applying filtered evaluation with all known true triples. To reduce GPU memory usage, the function directly scores only the candidate tail entities and computes realistic ranks.

In [15]:
import torch
import numpy as np
from collections import defaultdict

def evaluate_cve_to_cwe(model, test_triples, filter_triples, entities_subset, 
                        device, batch_size, slice_size):

    model = model.to(device)
    model.eval()

    if isinstance(filter_triples, torch.Tensor):
        filter_triples_np = filter_triples.detach().cpu().numpy()
    else:
        filter_triples_np = np.asarray(filter_triples)

    if isinstance(entities_subset, torch.Tensor):
        candidate_tails = entities_subset.to(device)
    else:
        candidate_tails = torch.tensor(entities_subset, dtype=torch.long, device=device)

    if candidate_tails.numel() == 0:
        raise ValueError('[ERROR] entities_subset is empty.')

    hr_to_true_tails = defaultdict(set)
    for h, r, t in filter_triples_np:
        hr_to_true_tails[(int(h), int(r))].add(int(t))

    candidate_tails_list = candidate_tails.detach().cpu().tolist()
    candidate_tail_to_pos = {entity_id: idx for idx, entity_id in enumerate(candidate_tails_list)}

    test_triples = test_triples.to(device)
    ranks = []

    with torch.inference_mode():
        total = len(test_triples)

        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            batch = test_triples[start:end]

            heads = batch[:, 0]
            rels = batch[:, 1]
            true_tails = batch[:, 2]

            hr_batch = torch.stack([heads, rels], dim=1)

            subset_scores = model.predict_t(hr_batch, tails=candidate_tails, slice_size=slice_size)

            for i in range(batch.shape[0]):
                h = int(heads[i].item())
                r = int(rels[i].item())
                t = int(true_tails[i].item())

                if t not in candidate_tail_to_pos:
                    raise ValueError(f'[ERROR] True tail entity ID {t} is not included in entities_subset.')

                scores_i = subset_scores[i].clone()
                true_tails_for_hr = hr_to_true_tails[(h, r)]

                filter_positions = [
                    candidate_tail_to_pos[cand_t]
                    for cand_t in true_tails_for_hr
                    if cand_t != t and cand_t in candidate_tail_to_pos
                ]

                if filter_positions:
                    filter_positions = torch.tensor(filter_positions, dtype=torch.long, device=device)
                    scores_i[filter_positions] = -torch.inf

                true_pos = candidate_tail_to_pos[t]
                true_score = scores_i[true_pos]

                optimistic_rank = 1 + torch.sum(scores_i > true_score).item()
                pessimistic_rank = torch.sum(scores_i >= true_score).item()
                realistic_rank = 0.5 * (optimistic_rank + pessimistic_rank)

                ranks.append(realistic_rank)

    return np.asarray(ranks, dtype=np.float64)

### 13. Evaluate CVE→CWE Triples Using the CWE Candidate List

Evaluate the trained TransE model on the CVE→CWE target triples in the cybersecurity knowledge graph. Since the MatchingCWE relation is stored as <CVE, MatchingCWE, CWE>, the evaluation is performed as a tail prediction task. The candidate entities are restricted to the CWE candidate list prepared during preprocessing, and filtered ranking is applied using all known true triples.

In [16]:
cwe_candidate_file = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12' / 'cwe_candidate_list.txt'

if not cwe_candidate_file.exists():
    raise FileNotFoundError(f'[ERROR] CWE candidate list not found: {cwe_candidate_file}')

with open(cwe_candidate_file, 'r', encoding='utf-8') as f:
    cwe_candidate_list = [line.strip() for line in f if line.strip()]

all_true_triples = torch.cat(
    [
        training_factory.mapped_triples,
        validation_factory.mapped_triples,
        testing_factory.mapped_triples,
    ],
    dim=0,
)

cwe_ids = sorted(
    training_factory.entity_to_id[cwe_name]
    for cwe_name in cwe_candidate_list
    if cwe_name in training_factory.entity_to_id
)

missing_cwe = [
    cwe_name
    for cwe_name in cwe_candidate_list
    if cwe_name not in training_factory.entity_to_id
]

print('=' * 40)
print('[INFO] Preparing CVE->CWE evaluation ...')
print(f'[INFO] CWE candidate list file: {cwe_candidate_file}')
print(f'[INFO] Number of CWE candidate names: {len(cwe_candidate_list):,}')
print(f'[INFO] Number of mapped CWE IDs: {len(cwe_ids):,}')
print(f'[INFO] Number of missing CWE names: {len(missing_cwe):,}')
print(f'[INFO] Number of CVE-CWE test triples: {len(test_cve_cwe_triples):,}')
print(f'[INFO] Number of filter triples: {len(all_true_triples):,}')
print('=' * 40)

ranks_cve_to_cwe = evaluate_cve_to_cwe(model_transe, test_cve_cwe_triples, all_true_triples, cwe_ids, 
                                        device='cuda', batch_size=128, slice_size=2048)

mrr = np.mean(1.0 / ranks_cve_to_cwe)
mr = np.mean(ranks_cve_to_cwe)
hits20 = np.mean(ranks_cve_to_cwe <= 20)
hits10 = np.mean(ranks_cve_to_cwe <= 10)
hits3 = np.mean(ranks_cve_to_cwe <= 3)
hits1 = np.mean(ranks_cve_to_cwe <= 1)

print('=' * 40)
print('[DONE] CVE->CWE evaluation completed successfully')
print(f'[INFO] MRR: {mrr:.3f}')
print(f'[INFO] MR: {int(mr):,}')
print(f'[INFO] Hits@20: {hits20:.3f}')
print(f'[INFO] Hits@10: {hits10:.3f}')
print(f'[INFO] Hits@3: {hits3:.3f}')
print(f'[INFO] Hits@1: {hits1:.3f}')
print('=' * 40)

[INFO] Preparing CVE->CWE evaluation ...
[INFO] CWE candidate list file: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\cwe_candidate_list.txt
[INFO] Number of CWE candidate names: 944
[INFO] Number of mapped CWE IDs: 944
[INFO] Number of missing CWE names: 0
[INFO] Number of CVE-CWE test triples: 135
[INFO] Number of filter triples: 1,705,929
[DONE] CVE->CWE evaluation completed successfully
[INFO] MRR: 0.319
[INFO] MR: 69
[INFO] Hits@20: 0.630
[INFO] Hits@10: 0.556
[INFO] Hits@3: 0.356
[INFO] Hits@1: 0.207
